# **<span style="color: black; font-size:1em;">IDAES-LCA Integration (ILI)</span>**

## <span style="color:black; font-weight:bold"> Introduction </span> 

<span style = "color:black">
This document demonstrates the integration of Life Cycle Assessment into the IDAES workflow. The goal is to develop a solution allowing a bi-directional communication between PrOMMiS and openLCA, and enabling a selective, constraint-based, environmental optimization framework 
</br>

<u> Goal and Scope:</u> In this Jupyter Notebook, we use life cycle assessment to evaluate the environmental impact of Rare Earth Oxide Recovery from coal mining refuse.</br>
As shown in the system boundary below, the scope of this work starts with the leaching of size-reduced REE-rich feedstock (REE: Rare Earth Elements) and ends with the recovery of mixed REO solids. The process consists of six main stages: 1) Mixing and Leaching, 2) Rougher Solvent Extraction, 3) Cleaner Solvent Extraction, 4) Precipitation, 5) Solid-Liquid (S/L) separation, and 6) Roasting. </br>
It's crucial to note here that the current script does not account for:

* Upstream processes leading to the production of REE-rich feedstock 


* Downstream processes leading to the separation of REE contained in the REO    

<u> Main Product:</u> The main product of the flowsheet modeled in this Jupyter Notebook is Rare Earth Oxide (REO) solid.

<u> Co-products or by-products:</u> None

<u> Functional Unit:</u> The function unit is 1 kg of recovered REO solids

<u> Allocation:</u> The evaluated flowsheet produces a single product with no intermediate co-products or by-product. As a result, there is no need for allocation in this modeling exercise.

<u> Life cycle inventory Estimation:</u> The material and energy inputs shown in the system boundary figure below have been shortlisted and estimated based on the UKy flowsheet output, [technical reports on the production on the production of REE from coal and coal by-products](https://www.osti.gov/servlets/purl/1569277), and other relevant literature.
</span>

<span style = "color:black">

**Useful notes, definitions, and links:**
- **PrOMMiS model:** Process Optimization and Modeling for Minerals Sustainability initiative, led by the U.S. Department of Energy (DOE), specifically under NETL (National Energy Technology Laboratory). The PrOMMiS source code can be accessed on github through the following repository link: [https://github.com/prommis/prommis/tree/main](https://github.com/prommis/prommis/tree/main)<br> PrOMMiS serves to optimize processing flowsheets. This Jupyter Notebook focuses on a flowsheet for the extraction of REE from coal refuse. 


- **UKy Flowsheet Code:** [https://github.com/prommis/prommis/blob/main/src/prommis/uky/uky_flowsheet.py](https://github.com/prommis/prommis/blob/main/src/prommis/uky/uky_flowsheet.py)

</span>

<div style="text-align: center;">
    <img src="images/system_boundary_1.png" width="1000"/>
</div>

## <span style="color:black; font-weight:bold"> Step 1: Import the necessary tools</span> 

In [24]:
from netlolca.NetlOlca import NetlOlca

import src.foqus_class as foqus_class
from src.foqus_class import NetlFoqus

## <span style="color:black; font-weight:bold"> Step 2: Create a New FOQUS Graph, setup output directory, and initiate UKy flowsheet  </span> 

In [25]:
nf = NetlFoqus()
m = nf.init_uky()

INFO Created /home/franc/.netl
WARNING Install Path Does Not Exist: C:/Program Files/Turbine/Lite
Scaling fs.leach
Scaling fs.sl_sep1
Scaling fs.scrubber_HCl_leach_translator
Scaling fs.leach_mixer
Scaling fs.leach_liquid_feed
Scaling fs.leach_solid_feed
Scaling fs.leach_filter_cake
Scaling fs.leach_filter_cake_liquid
Scaling fs.rougher_org_make_up
Scaling fs.solex_rougher_load
Scaling fs.acid_feed1
Scaling fs.solex_rougher_scrub
Scaling fs.acid_feed2
Scaling fs.solex_rougher_strip
Scaling fs.rougher_sep
Scaling fs.rougher_mixer
Scaling fs.load_sep
Scaling fs.scrub_sep
Scaling fs.rougher_organic_purge
Scaling fs.solex_cleaner_load
Scaling fs.solex_cleaner_strip
Scaling fs.cleaner_org_make_up
Scaling fs.cleaner_mixer
Scaling fs.cleaner_sep
Scaling fs.cleaner_HCl_leach_translator
Scaling fs.leach_sx_mixer
Scaling fs.acid_feed3
Scaling fs.cleaner_organic_purge
Scaling fs.precipitator
Scaling fs.sl_sep2
Scaling fs.precip_sep
Scaling fs.precip_sx_mixer
Scaling fs.precip_purge
Scaling fs.roa

## <span style="color:black; font-weight:bold"> Step 3: FOQUS - Setup the PrOMMiS and openLCA Nodes </span> 

#### <span style="color:black; font-weight:bold"> Select the Decision Variables </span> 

<span style="color:black">
In the following example, two variables are chosen to optimize (i.e., `my_var_1` and `my_var_2`). These are added to the NetlFoqus Python class's list of decision variables (appended to the classes `dv` class attribute) and set as input variables to the PrOMMiS node. The value of each decision variable and their min/max amounts are set using the `initialize_decision_variables` method. For convenicne, these are saved to a CSV file in the output folder defined above.</span>

In [26]:
# Add a decision variable
my_var1 = "fs.leach_liquid_feed.flow_vol"
nf.add_decision_variable(my_var1)

# [FH] we need at least two decision variables
my_var2 = "fs.load_sep.split_fraction"
nf.add_decision_variable(my_var2)

# Help with initializing decision variables
foqus_class.initialize_decision_variables(nf, m)

# Store decision variable information in a dataframe and save to output directory
dv_df = foqus_class.export_decision_variables(nf)

# set decesion variables as input variables for prommis_node
for dv in nf.dv:
    nf.set_input_variables(nf.prommis_node, dv.ipvname, dv.value, dv.min, dv.max)

INFO Set decision properties for fs.leach_liquid_feed.flow_vol: value=224.300000, min=201.870000, max=246.730000
INFO Set decision properties for fs.load_sep.split_fraction: value=0.900000, min=0.810000, max=0.990000
INFO Set input variable fs.leach_liquid_feed.flow_vol for node ProMMiS: value=224.300000
INFO Set input variable fs.load_sep.split_fraction for node ProMMiS: value=0.900000


#### <span style="color:black; font-weight:bold"> Initialize the intermediate variables </span> 

<span style="color:black">
This step initializes starting values of the exchanges between the PrOMMiS node and the openLCA node and connects the parameters from one node to the other.</span>

In [27]:
# Initialize intermediate variables
nf.initialize_intermediate_variables(nf.prommis_node, nf.olca_node)

# connect intermediate variables
nf.connect_intermediate_variables(nf.prommis_node, nf.olca_node)

# save exchanges in nf.exchanges
lca_df_finalized = nf.exchanges

INFO Set output properties for 374 ppm REO Feed: value=1088399.130945
INFO Set output properties for 50% Caustic Solution: value=21048.466248
INFO Set output properties for 73.4% REO Product: value=1.000000
INFO Set output properties for Ascorbic Acid: value=1722.147238
INFO Set output properties for Carbon dioxide emissions: value=0.711129
INFO Set output properties for DEHPA: value=29034.809257
INFO Set output properties for Electricity, AC, 120 V: value=1247.526530
INFO Set output properties for Hydrochloric Acid: value=2876.160852
INFO Set output properties for Kerosene: value=487979.987505
INFO Set output properties for Natural Gas: value=3512.119790
INFO Set output properties for Nitrogen emissions: value=7.816932
INFO Set output properties for Oxalic Acid: value=23246.161221
INFO Set output properties for Oxygen emissions: value=1.340621
INFO Set output properties for Solid Waste: value=1067117.833934
INFO Set output properties for Sulfuric Acid: value=52737.772907
INFO Set outp

#### <span style="color:black; font-weight:bold"> Add additional PrOMMiS model outputs </span> 
<span style="color:black">
Below, a selection of ProMMiS model outputs are added to the model.
These parameters will then be available for optimization and/or contraints to the model as described later on in this notebook.</span> 

In [28]:
prommis_outputs_df = foqus_class.generate_prommis_outputs(nf, m)
prommis_outputs_df

INFO Initiated output variable total plant cost for node ProMMiS: value=1.180711
INFO Initiated output variable total bare erected cost for node ProMMiS: value=0.397543
INFO Initiated output variable total annualized capital cost for node ProMMiS: value=0.135343
INFO Initiated output variable total fixed OM cost for node ProMMiS: value=6.816580
INFO Initiated output variable total variable OM cost for node ProMMiS: value=1.363409
INFO Initiated output variable total OM cost for node ProMMiS: value=8.179989
INFO Initiated output variable total annualized plant cost for node ProMMiS: value=8.315333
INFO Initiated output variable anual rate of recovery for node ProMMiS: value=0.168056
INFO Initiated output variable cost of recovery per REE for node ProMMiS: value=49479526.539687
INFO Initiated output variable recovery rate for node ProMMiS: value=0.171973
INFO Initiated output variable product purity for node ProMMiS: value=62.909450


,output,value
0,total plant cost,1.180711e+00
1,total bare erected cost,3.975432e-01
2,total annualized capital cost,1.353435e-01
3,total fixed OM cost,6.816580e+00
4,total variable OM cost,1.363409e+00
5,total OM cost,8.179989e+00
6,total annualized plant cost,8.315333e+00
7,anual rate of recovery,1.680560e-01
8,cost of recovery per REE,4.947953e+07
9,recovery rate,1.719728e-01


#### <span style="color:black; font-weight:bold"> Initialize the LCA model </span> 

##### <u><span style="color:black; font-weight:bold">Connect to IPC</span></u>

In [29]:
netl = NetlOlca()
netl.connect()
netl.read()

INFO Connected on http://localhost:8080
INFO Read UUIDs from IPC connection.


##### <u><span style="color:black; font-weight:bold">Enter Process Name and Description</span></u>

In [30]:
process_name = "TESTING - REO Extraction From Coal Mining Refuse | UKy Flowsheet"

process_description = """This process involves the production of a Rare 
Earth Oxide solid extraction from coal mining refuse. The scope of this 
work starts with the leaching of size-reduced REE-rich feedstock (REE: 
Rare Earth Elements) and ends with the recovery of mixed REO solids. The 
process consists of six main stages: 1) Mixing and Leaching, 2) Rougher 
Solvent Extraction, 3) Cleaner Solvent Extraction, 4) Precipitation, 
5) Solid-Liquid (S/L) separation, and 6) Roasting. This process does not 
account for upstream processes leading to the production of REE-rich 
feedstock nor does it account for Downstream processes leading to the 
separation of REE contained in the REO. The main product is a rare earth 
oxide solid with no other by-products or co-products. The functional 
unit is 1 kg of recovered REO solids. The material and energy inputs 
shown in the system boundary figure below have been shortlisted and 
estimated based on the UKy flowsheet output, as well as other relevant 
literature."""

##### <u><span style="color:black; font-weight:bold">Select Impact Assessment Method</span></u>

<span style="color:black">
The following cell includes the UUID of the impact assessment method to be used in subsequent steps.
The UUID in the below cell refers to TRACI 2.1 (NETL)</span>

In [31]:
impact_method_uuid = '60cb71ff-0ef0-4e6c-9ce7-c885d921dd15'

##### <u><span style="color:black; font-weight:bold">Configure Parameter Set</span></u>

In [32]:
parameter_set_name = "Baseline"
parameter_set_description = "Baseline parameter set for the process"
is_baseline = True

##### <u><span style="color:black; font-weight:bold">Run first step in openLCA</span></u>

<span style="color:black">
The following cell includes the following steps in openLCA:

* Create a new process
* Assign parameters to each exchange
* Create a new product system
* Create a parameter set
* Run analysis using the selected impact caategory
* Export the parameters, impacts, and run information if the save_outputs = True
</span>

In [33]:
total_impacts, my_parameters, ps_uuid = foqus_class.initiate_lca_model (netl, 
                                process_name, 
                                process_description, 
                                lca_df_finalized, 
                                impact_method_uuid, 
                                parameter_set_name, 
                                parameter_set_description, 
                                is_baseline,
                                save_outputs=True,
                                output_dir=nf.output_dir)

Creating exchange database, this may take a couple minutes...


Creating exchange for product flow: 374 ppm REO Feed
-----------------------------------
Searching for flows containing '374 ppm REO Feed'...
No flows found matching '374 ppm REO Feed'
No flows found matching the criteria.
ERROR no ID or name given
Error creating exchange for product flow: Flow not found: None


Creating exchange for product flow: 50% Caustic Solution
-----------------------------------
Searching for flows containing 'sodium hydroxide'...
Found 5 flows matching 'sodium hydroxide'
Filtered to 1 PRODUCT_FLOW flows
  1. Flow_Name: Sodium hydroxide; at plant | UUID: d12c4a32-12a8-4673-9b14-5d227d2a128e
  1. Process_Name: Chlorine; chlor-alkali electrolysis; at plant | Process_UUID: a3e150d0-770e-4e2a-9b19-f7daa8cda38b
  2. Process_Name: Sodium hydroxide; chlor-alkali average, membrane cell; at plant; 50% solution state | Process_UUID: 344a7ac0-8c18-4869-bb63-3e44e811a1d9
Selected process UUID: a3e150d0-770e-4e

##### <u><span style="color:black; font-weight:bold">Create output variables of openLCA node in FOQUS</span></u>
<span style="color:black">
The results from the first openLCA run are used initialize output values from the openLCA node. </br>
This will allow the solver access the openLCA results.</span>

In [34]:
foqus_class.create_openlca_outputs(nf, total_impacts, nf.olca_node)

INFO Initiated output variable Water Consumption (NETL) for node openLCA: value=64246389.148406
INFO Initiated output variable Cumulative Energy Demand for node openLCA: value=44526737.886017
INFO Initiated output variable Global Warming Potential [AR6, 20 yr] for node openLCA: value=1122907.086956
INFO Initiated output variable Global Warming Potential [AR6, 100 yr] for node openLCA: value=1024283.128475
INFO Initiated output variable Cumulative Energy Demand - Renewable for node openLCA: value=5047551.176278
INFO Initiated output variable Cumulative Energy Demand - Non-renewable for node openLCA: value=25804900.427208
INFO Initiated output variable Smog formation for node openLCA: value=192425.977119
INFO Initiated output variable Freshwater ecotoxicity for node openLCA: value=2093263.933676
INFO Initiated output variable Human health - cancer for node openLCA: value=0.001382
INFO Initiated output variable Eutrophication for node openLCA: value=5095.899966
INFO Initiated output varia

##### <u><span style="color:black; font-weight:bold">Define openLCA and PrOMMiS node scripts</span></u>
<span style="color:black">
Below shows the node scripts that were created for running FOQUS. The user should not edit these scripts. </br>
Below a brief description of the functionality of each script.

<u>PrOMMiS node script</u>:
* The PrOMMiS decision variables, also defined as inputs to the PrOMMiS node, are initiated with values in previous steps of this jupyter notebook.
* The initial decision variables values are retrieved and used to build the UKy coal refuse model
* The PrOMMiS flowsheet is evaluated, yielding flosheet design parameters (flow volume, elemental mass concentration, etc.)
* The PrOMMiS raw data is converted to LCA-relevant units and assigned to the respective PrOMMiS node outputs
* The PrOMMiS node outputs are transferred to the openLCA node

<u>openLCA node script</u>:
* The openLCA node inputs are populated with the exchanges' values from the PrOMMiS node
* The exchanges values are retrieved and populated in openLCA to their respective parameters in the pre-defined parameter set.
* The product system is re-evaluated using the updated parameter set.
* The analysis results are exported and assigned to their respective openLCA node output variables

<u>Iteration process</u>:
* The openLCA results (e.g., Global Warming Potential) are later passed to the solver
* Accordingly the solver updates the input decision variables in the PrOMMiS node to be re-evaluated in a subsequent iteration

</span>

In [35]:
#openLCA node
nf.define_node_script(nf.olca_node, foqus_class.openlca_node_script)

#PrOMMiS node
nf.define_node_script(nf.prommis_node, foqus_class.prommis_node_script)

INFO Defined script for node openLCA: script=
# import dependencies
from pathlib import Path
import pandas as pd
import olca_schema as olca
import re
import os

from netlolca.NetlOlca import NetlOlca

import src as lca_prommis

working_dir = os.path.join(os.path.expanduser("~"), ".netl")
if os.path.isdir(working_dir):
    output_dir = working_dir
else:
    output_dir = os.path.expanduser("~")

# get the parameters file from the output directory
my_parameters_path = os.path.join(output_dir, "my_parameters.csv")
my_parameters = pd.read_csv(my_parameters_path)
params = my_parameters.copy() # get a copy of the parameters df

# update the parameters table with the new parameter values
existing = [
    int(m.group(1))
    for col in params.columns
    if (m := re.match(r"parameter_value_(\d+)", col))
]
new_col = f"parameter_value_{max(existing, default=0) + 1}"
# update the parameters table with the new parameter values
for count, row in params.iterrows():
    desc = row['parameter_descripti

## <span style="color:black; font-weight:bold"> Step 4: Create Session </span> 

In [36]:
# Enter your foqus working directory
my_session = nf.create_session()

INFO Working directory is already /home/franc/foqus_wd
WARNING Install Path Does Not Exist: C:/Program Files/CCSI/SimSinter
WARNING Install Path Does Not Exist: C:/Program Files/Turbine/Lite
WARNING Install Path Does Not Exist: C:/Program Files/Turbine/Lite
WARNING Install Path Does Not Exist: C:/Program Files/Turbine/Lite
2026-03-12 16:22:33,094 - DEBUG - foqus.foqus_lib.framework.session.session - Initializing session, Log file: /home/franc/foqus_wd/logs/foqus.log, Position: 2919237
DEBUG Initializing session, Log file: /home/franc/foqus_wd/logs/foqus.log, Position: 2919237
2026-03-12 16:22:33,096 - INFO - foqus.foqus_lib.framework.plugins.pluginSearch - Loading Plugin: /home/franc/FOQUS/foqus_lib/framework/surrogate/ALAMO.py
INFO Loading Plugin: /home/franc/FOQUS/foqus_lib/framework/surrogate/ALAMO.py
2026-03-12 16:22:33,098 - INFO - foqus.foqus_lib.framework.plugins.pluginSearch - Loading Plugin: /home/franc/FOQUS/foqus_lib/framework/surrogate/BSS-ANOVA.py
INFO Loading Plugin: /hom

## <span style="color:black; font-weight:bold"> Step 5: Create a problem and setup optimizer </span> 
<span style="color:black">
The first step in setting up the optimizer is defining the problem, which consists of: 

- The working session
- The optimization method ("NLopt")
- The node with the input variables to be optimized</span> 

In [37]:
problem = nf.setup_optimizer(my_session, "NLopt", nf.prommis_node) # first step in setting up optimizer


## <span style="color:black; font-weight:bold"> Step 6: Create Problem Objective </span> 

<span style="color:black">

Setting an objective requires a set of inputs including the following:

* The objective expression representing the amount to be minimized
* A penalty scale: This is most needed when there are multiple objectives and at least one constraint. The purpose of the penalty scale is to help the solver apply the constraint violation penalty differently to each different objective function.
* Value for failure:  value to be assigned to the objective function if the objective cannot be evaluated for any reason. The value should be higher than the expected highest value for a successful objective. 

There are different approaches to setting an objective:

* Approach 1: Assigning multiple single objectives. One example could involve creating two separate objectives: 1) Global Warming Potential and 2) Freshwater ecotoxicity. The solution in this case will tend to converge towards minimizing both objectives. A defined penalty scale is generally needed in this case, the following cells include an automated method to generate a penalty factor for each selected objective. 
* Approach 2: Assigning a single objective function that covers multiple variables. This will require the user the input a set of weights for each of the variables to be included in the function - ideally these weights must add up to 1. The developed methods for this approach would automatically apply factors to properly scale all the selected variables. 
One example could invovle the following variables in a single function: Total capital cost with a weight of 0.3, freshwater ecotoxicity with a weight of 0.4, and global warming potential with a weight of 0.3.

Instructions to set an objective:

* Please use one of the next two cells 6.1 and 6.2 to setup your objective. The cells are already setup to depict each of the examples described above
* Variables selection options: Please run the next cell for a list of variables that can be used in the objective setup function. Please ensure the exact same variable names are used in cells 6.1 and 6.2.

</span>

##### <u><span style="color:black; font-weight:bold">Available variables</span></u>

In [38]:
nf.outVars

{'ProMMiS': ['374 ppm REO Feed',
  '50% Caustic Solution',
  '73.4% REO Product',
  'Ascorbic Acid',
  'Carbon dioxide emissions',
  'DEHPA',
  'Electricity, AC, 120 V',
  'Hydrochloric Acid',
  'Kerosene',
  'Natural Gas',
  'Nitrogen emissions',
  'Oxalic Acid',
  'Oxygen emissions',
  'Solid Waste',
  'Sulfuric Acid',
  'Wastewater',
  'Water',
  'Water emissions',
  'total plant cost',
  'total bare erected cost',
  'total annualized capital cost',
  'total fixed OM cost',
  'total variable OM cost',
  'total OM cost',
  'total annualized plant cost',
  'anual rate of recovery',
  'cost of recovery per REE',
  'recovery rate',
  'product purity'],
 'openLCA': ['Water Consumption (NETL)',
  'Cumulative Energy Demand',
  'Global Warming Potential [AR6, 20 yr]',
  'Global Warming Potential [AR6, 100 yr]',
  'Cumulative Energy Demand - Renewable',
  'Cumulative Energy Demand - Non-renewable',
  'Smog formation',
  'Freshwater ecotoxicity',
  'Human health - cancer',
  'Eutrophication',

##### <u><span style="color:black; font-weight:bold">6.1. Single-variable multiple-objectives setup</span></u>

In [39]:
objectives_list = ["Freshwater ecotoxicity", "Global Warming Potential [AR6, 100 yr]"]

In [40]:
ps_guide = foqus_class.generate_penalty_scales(prommis_outputs_df, total_impacts)
penalty_scale_list = foqus_class.get_penalty_scales(objectives_list, ps_guide)

In [41]:
problem = nf.create_problem_objective_singular(problem,                                         
                                            objectives_list, 
                                            [nf.olca_node, nf.olca_node],
                                            [10000000, 1000000], #dummy values
                                            penalty_scale_list
                                            ) 

##### <u><span style="color:black; font-weight:bold">6.2. Multi-variable single objectives setup</span></u>

In [ ]:
objectives_list = ["total plant cost", "Freshwater ecotoxicity", "Global Warming Potential [AR6, 100 yr]"]

In [ ]:
ps_guide = foqus_class.generate_penalty_scales(prommis_outputs_df, total_impacts)
penalty_scale_list = foqus_class.get_penalty_scales(prommis_outputs_df, ps_guide)

In [ ]:
problem = nf.create_problem_objective_multiple(problem,                                         
                                            objectives_list, 
                                            [nf.prommis_node, nf.olca_node, nf.olca_node],
                                            [1.3, 10000000, 1000000], #dummy values
                                            penalty_scale_list,
                                            [0.3, 0.4, 0.3]
                                            ) 

## <span style="color:black; font-weight:bold"> Step 7: Create Problem Constraint <i>(Optional)</i> </span> 

<span style="color:black">

Backroung, context, and instructions to setup a problem constraint:

* Contraint contextualization: Setting up a constraint helps exclude potential solutions that meet the objective but violate certain conditions such as the operating cost or the REE recovery rate. 

* The user has to define an inquality contraint function g(x) of the form:
    g(x) < 0

* If the constraint is violated, the solver generates a penalty that is applied to the objective. This penalty depends on two user inputs: 1) penalty factor and 2) penalty form. The effect of the penalty factor and penalty form can be show in the Equations (1-3) below. <i><b><u>Source</u></b>: [FOQUS Documentation - Optimization](https://foqus.readthedocs.io/en/stable/chapt_opt/reference/optimization.html)</i>

* <b><u>Example</b></u>: Contraint to set a minimum recovery rate of 90%. In this case the constraint function g(x) = 0.9 - (Recovery Rate).

* Instructions:
    * If the constraint consists of setting a maximum value for a variable, please use <b>nf.get_max_constraint</b> under section 7.1
    * If the constraint consists of setting a minimum value for a variable, please use <b>nf.get_min_constraint</b> under section 7.1
    * After creating the constraint function, proceed to section 7.2 in order to assign the constraint to the problem

<span style = "color:red"><b><u>Important Note:</b></u></span> 

* Outputs used to define decision variables or problem objectives cannot be used as constraints.
* If you want to set more than one constraint (for example a 2nd constraint), create a new function "g_x_2" and assign it to the problem using the "create_problem_constraint" method, which will append the 2nd constraint and not overwrite any previosuly set constraints.
* Run the cell under 'Available Variables' to get a list of variables with their respective nodes, that can be used to create a constraint

</span>

<div style="text-align: center;">
    <img src="images/penalty_forms.png" width="650"/>
</div>

##### <u><span style="color:black; font-weight:bold">Available variables</span></u>

In [42]:
nf.outVars

{'ProMMiS': ['374 ppm REO Feed',
  '50% Caustic Solution',
  '73.4% REO Product',
  'Ascorbic Acid',
  'Carbon dioxide emissions',
  'DEHPA',
  'Electricity, AC, 120 V',
  'Hydrochloric Acid',
  'Kerosene',
  'Natural Gas',
  'Nitrogen emissions',
  'Oxalic Acid',
  'Oxygen emissions',
  'Solid Waste',
  'Sulfuric Acid',
  'Wastewater',
  'Water',
  'Water emissions',
  'total plant cost',
  'total bare erected cost',
  'total annualized capital cost',
  'total fixed OM cost',
  'total variable OM cost',
  'total OM cost',
  'total annualized plant cost',
  'anual rate of recovery',
  'cost of recovery per REE',
  'recovery rate',
  'product purity'],
 'openLCA': ['Water Consumption (NETL)',
  'Cumulative Energy Demand',
  'Global Warming Potential [AR6, 20 yr]',
  'Global Warming Potential [AR6, 100 yr]',
  'Cumulative Energy Demand - Renewable',
  'Cumulative Energy Demand - Non-renewable',
  'Smog formation',
  'Freshwater ecotoxicity',
  'Human health - cancer',
  'Eutrophication',

##### <u><span style="color:black; font-weight:bold">7.1. Generate constraint function </span></u>

##### <u><span style="color:black; font-weight:bold">Generating constraint to set maximum variable value</span></u>

In [43]:
g_x = nf.get_max_constraint(variable = "Water Consumption (NETL)", 
                            node = nf.olca_node, 
                            max_value = 1000
                            )

##### <u><span style="color:black; font-weight:bold">Generating constraint to set minimum variable value</span></u>

In [44]:
g_x = nf.get_min_constraint(variable = "recovery rate", 
                            node = nf.prommis_node, 
                            min_value = 0.2
                            )

##### <u><span style="color:black; font-weight:bold">7.2. Assign constraint to the problem </span></u>

In [45]:
problem = nf.create_problem_constraint (problem, 
                                        pycode = g_x, 
                                        penalty_factor = 10, 
                                        form = 'Linear'
                                        )  

## <span style="color:black; font-weight:bold"> Step 8: Setup solver options </span> 

In [46]:
problem = nf.setup_nlopt_solver_options(problem, True) 

## <span style="color:black; font-weight:bold"> Step 9: Run Optimization </span> 

In [47]:
my_solver, problem = nf.run_optimization(problem, my_session) 

WARNING Install Path Does Not Exist: C:/Program Files/Turbine/Lite
2026-03-12 16:23:02,574 - DEBUG - foqus.foqus_lib.framework.graph.graph - run: Running flowsheet(s) locally
DEBUG run: Running flowsheet(s) locally
2026-03-12 16:23:02,578 - DEBUG - foqus.foqus_lib.framework.graph.graph - run: solve list simulations
DEBUG run: solve list simulations
2026-03-12 16:23:02,579 - DEBUG - foqus.foqus_lib.framework.graph.graph - solve: runGraph
DEBUG solve: runGraph
INFO Leach liquid feed: 224.300000
INFO split_fraction: 0.900000
Scaling fs.leach
Scaling fs.sl_sep1
Scaling fs.scrubber_HCl_leach_translator
Scaling fs.leach_mixer
Scaling fs.leach_liquid_feed
Scaling fs.leach_solid_feed
Scaling fs.leach_filter_cake
Scaling fs.leach_filter_cake_liquid
Scaling fs.rougher_org_make_up
Scaling fs.solex_rougher_load
Scaling fs.acid_feed1
Scaling fs.solex_rougher_scrub
Scaling fs.acid_feed2
Scaling fs.solex_rougher_strip
Scaling fs.rougher_sep
Scaling fs.rougher_mixer
Scaling fs.load_sep
Scaling fs.scru

/home/franc/FOQUS/foqus_lib/framework/sampleResults/results.py:436: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  self.loc[row, col] = dat[i]


INFO Leach liquid feed: 235.515000
INFO split_fraction: 0.900000
Scaling fs.leach
Scaling fs.sl_sep1
Scaling fs.scrubber_HCl_leach_translator
Scaling fs.leach_mixer
Scaling fs.leach_liquid_feed
Scaling fs.leach_solid_feed
Scaling fs.leach_filter_cake
Scaling fs.leach_filter_cake_liquid
Scaling fs.rougher_org_make_up
Scaling fs.solex_rougher_load
Scaling fs.acid_feed1
Scaling fs.solex_rougher_scrub
Scaling fs.acid_feed2
Scaling fs.solex_rougher_strip
Scaling fs.rougher_sep
Scaling fs.rougher_mixer
Scaling fs.load_sep
Scaling fs.scrub_sep
Scaling fs.rougher_organic_purge
Scaling fs.solex_cleaner_load
Scaling fs.solex_cleaner_strip
Scaling fs.cleaner_org_make_up
Scaling fs.cleaner_mixer
Scaling fs.cleaner_sep
Scaling fs.cleaner_HCl_leach_translator
Scaling fs.leach_sx_mixer
Scaling fs.acid_feed3
Scaling fs.cleaner_organic_purge
Scaling fs.precipitator
Scaling fs.sl_sep2
Scaling fs.precip_sep
Scaling fs.precip_sx_mixer
Scaling fs.precip_purge
Scaling fs.roaster
Scaling fs.leaching_sol_feed

## <span style="color:black; font-weight:bold"> Step 10: Extract Results </span> 

In [ ]:
parameters_s, impacts_s, dv_s, prommis_s = foqus_class.get_optimization_results(netl, 
                                                                                ps_uuid, 
                                                                                parameter_set_name, 
                                                                                solver = my_solver, 
                                                                                decision_variables = dv_df, 
                                                                                prommis_outputs = prommis_outputs_df, 
                                                                                parameters = my_parameters, 
                                                                                total_impacts = total_impacts)